# Lab 07 — Windup do integrador e ciclos-limite: o experimento do relé

**Unidade III — Análise de não-linearidades em malhas de controle** · conteúdo 3.1 do PPC

**Objetivos:**
1. Dissecar o mecanismo do **windup**: observar o estado do integrador durante a saturação;
2. Implementar e validar o **anti-windup por back-calculation**;
3. Executar o **experimento do relé** (Åström–Hägglund) e extrair $K_u$ e $T_u$ do ciclo-limite;
4. Entregar os parâmetros críticos que alimentarão a sintonia da Unidade IV.

**Referências:** Åström & Murray (FBS), cap. 11 · Åström & Hägglund, *Advanced PID Control*, caps. 3–4.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
try:
    import control as ct
    print("python-control", ct.__version__)
except ImportError:
    %pip install control
    import control as ct

## 1. Anatomia do windup

Para **ver** o estado do integrador, implementamos o controlador PI como `ct.nlsys` com o
integrador como estado explícito, incluindo a saturação **dentro** do controlador:

$$\dot{x}_i = \frac{K_p}{T_i}e + \frac{1}{T_t}\big(u_{sat} - u\big) \qquad
  u = K_p e + x_i \qquad u_{sat} = \mathrm{sat}(u)$$

Com $T_t \to \infty$ o termo de correção desaparece (PI **sem** anti-windup);
com $T_t$ finito temos o anti-windup por *back-calculation*.

In [ ]:
# planta: motor CC reduzido
G = ct.tf([5], [2, 1])
plant = ct.tf2io(G, inputs='u', outputs='y', name='plant')

Kp, Ti = 0.8, 1.5
UMAX = 0.6   # saturação severa (mesma do Lab 06)

def pi_update(t, x, u, params):
    """Estado x[0] = parcela integral. Entradas: u = [e]."""
    e = u[0]
    v = params['Kp'] * e + x[0]                       # controle pedido
    u_sat = np.clip(v, -params['umax'], params['umax'])  # controle aplicado
    aw = (u_sat - v) / params['Tt'] if np.isfinite(params['Tt']) else 0.0
    return [params['Kp'] / params['Ti'] * e + aw]

def pi_output(t, x, u, params):
    """Saídas: [u_sat, v, xi] — controle aplicado, pedido e estado integral."""
    e = u[0]
    v = params['Kp'] * e + x[0]
    u_sat = np.clip(v, -params['umax'], params['umax'])
    return [u_sat, v, x[0]]

def make_pi(Tt):
    return ct.nlsys(pi_update, pi_output, states=1,
                    inputs='e', outputs=['u', 'v', 'xi'],
                    params={'Kp': Kp, 'Ti': Ti, 'umax': UMAX, 'Tt': Tt},
                    name='pi')

summer = ct.summing_junction(inputs=['r', '-y'], output='e', name='sum')

def build_loop(Tt):
    return ct.interconnect([summer, make_pi(Tt), plant],
                           inputs='r', outputs=['y', 'u', 'v', 'xi'])

In [ ]:
t = np.linspace(0, 25, 2500)
r = 4.0 * np.ones_like(t)

resp_wind = ct.input_output_response(build_loop(np.inf), t, r)   # sem anti-windup
Tt = np.sqrt(Ti * 0.1)                                            # regra prática
resp_aw = ct.input_output_response(build_loop(Tt), t, r)          # com anti-windup

fig, axs = plt.subplots(3, 1, figsize=(9, 9), sharex=True)
axs[0].plot(resp_wind.time, resp_wind.outputs[0], 'C3', lw=2, label='sem anti-windup')
axs[0].plot(resp_aw.time, resp_aw.outputs[0], 'C2', lw=2, label='com anti-windup')
axs[0].axhline(4, color='gray', ls='--')
axs[0].set_ylabel('Saída y'); axs[0].legend(); axs[0].grid(True)

axs[1].plot(resp_wind.time, resp_wind.outputs[2], 'C3--', lw=1.2, label='v pedido (sem AW)')
axs[1].plot(resp_wind.time, resp_wind.outputs[1], 'C3', lw=2, label='u aplicado (sem AW)')
axs[1].plot(resp_aw.time, resp_aw.outputs[1], 'C2', lw=2, label='u aplicado (com AW)')
axs[1].set_ylabel('Controle'); axs[1].legend(); axs[1].grid(True)

axs[2].plot(resp_wind.time, resp_wind.outputs[3], 'C3', lw=2, label='integrador (sem AW)')
axs[2].plot(resp_aw.time, resp_aw.outputs[3], 'C2', lw=2, label='integrador (com AW)')
axs[2].set_ylabel('Estado integral $x_i$'); axs[2].set_xlabel('Tempo [s]')
axs[2].legend(); axs[2].grid(True)
fig.suptitle('Windup dissecado: o integrador é o culpado')
plt.show()

**Leitura dos gráficos:** sem anti-windup, o integrador (painel 3) cresce muito além do necessário
enquanto o atuador está saturado — e demora a "desenrolar", causando o sobressinal do painel 1.
Com back-calculation, o integrador é freado assim que $u \ne v$, e a resposta melhora
drasticamente **sem alterar a sintonia**.

## 2. Ciclos-limite úteis: o experimento do relé

Fechando a malha com um **relé** no lugar do controlador, quase toda planta industrial entra em
ciclo-limite estável. Åström & Hägglund mostraram que esse ciclo revela o ponto crítico:

$$K_u \approx \frac{4d}{\pi a} \qquad \omega_u = \frac{2\pi}{T_u}$$

onde $d$ é a amplitude do relé, $a$ a amplitude da oscilação de saída e $T_u$ o período medido.
É a alternativa **segura** a levar a malha ao limiar de instabilidade (Lab 05).

In [ ]:
# planta de 3ª ordem do Lab 05: sabemos que Ku = 90 e Tu = 1.68 s (gabarito analítico)
G3 = ct.tf([1], np.polymul(np.polymul([1, 1], [1, 2]), [1, 4]))

DT = 0.001   # passo da simulação discreta do relé
plant3 = ct.tf2io(ct.sample_system(G3, DT), inputs='u', outputs='y', name='plant')

def relay_update(t, x, u, params):
    """Relé com pequena histerese para evitar chattering numérico."""
    e, h = u[0], params['h']
    x_new = x[0]
    if e > h:
        x_new = params['d']
    elif e < -h:
        x_new = -params['d']
    return [x_new]

relay = ct.nlsys(relay_update, lambda t, x, u, params: x[0],
                 states=1, dt=DT, inputs='e', outputs='u',
                 params={'d': 1.0, 'h': 0.001}, name='relay')

summer3 = ct.summing_junction(inputs=['r', '-y'], output='e', name='sum')
relay_loop = ct.interconnect([summer3, relay, plant3], inputs='r', outputs=['y', 'u'])

t_r = np.arange(0, 20, DT)
r_r = np.zeros_like(t_r)   # referência nula: oscilação em torno de zero
resp_r = ct.input_output_response(relay_loop, t_r, r_r, X0=[1.0, 0, 0, 0])

fig, axs = plt.subplots(2, 1, figsize=(9, 6), sharex=True)
axs[0].plot(resp_r.time, resp_r.outputs[0], lw=1.5)
axs[0].set_ylabel('Saída y'); axs[0].grid(True)
axs[1].step(resp_r.time, resp_r.outputs[1], lw=1)
axs[1].set_ylabel('Relé u'); axs[1].set_xlabel('Tempo [s]'); axs[1].grid(True)
fig.suptitle('Experimento do relé: ciclo-limite estável')
plt.show()

In [ ]:
# medição automática de a e Tu no trecho de regime (últimos 10 s)
sel = resp_r.time > 10
y_ss = resp_r.outputs[0][sel]
t_ss = resp_r.time[sel]

a = (y_ss.max() - y_ss.min()) / 2                     # amplitude da oscilação

# período: intervalo médio entre cruzamentos ascendentes de zero
sinal = np.sign(y_ss)
cruzamentos = t_ss[1:][(sinal[1:] > 0) & (sinal[:-1] <= 0)]
T_u = np.mean(np.diff(cruzamentos))

d = 1.0
K_u = 4 * d / (np.pi * a)
print(f"Medido do ciclo-limite: a = {a:.4f}, T_u = {T_u:.3f} s")
print(f"Estimativas: K_u = 4d/(pi a) = {K_u:.1f}  |  omega_u = {2*np.pi/T_u:.2f} rad/s")
print("Gabarito analítico (Lab 05): K_u = 90, omega_u = 3.74 rad/s, T_u = 1.68 s")

O experimento do relé recupera o ponto crítico com erro pequeno (a aproximação da função
descritiva ignora harmônicos). **Anote $(K_u, T_u)$: são a entrada direta das tabelas de
Ziegler–Nichols no Lab 09** — e o mesmo procedimento será executado na planta física do
projeto final.

### 2.1 O ciclo-limite visto no plano de fase

Complementando os retratos de fase do Lab 06: projetando a trajetória em duas coordenadas de
estado, o ciclo-limite aparece como uma **órbita fechada atratora** — trajetórias de dentro e
de fora convergem para ela. É a assinatura geométrica que distingue um ciclo-limite
(amplitude própria, robusta) de oscilações lineares marginais (amplitude dependente da
condição inicial):

In [ ]:
# reconstrução aproximada de (y, dy/dt) a partir da simulação do relé
y_full = resp_r.outputs[0]
dy_full = np.gradient(y_full, DT)

plt.figure(figsize=(7, 6))
plt.plot(y_full, dy_full, lw=0.8, color='C0', alpha=0.9)
plt.plot(y_full[sel][-2000:], dy_full[sel][-2000:], lw=2.5, color='k',
         label='órbita de regime (ciclo-limite)')
plt.plot(y_full[0], dy_full[0], 'go', label='condição inicial')
plt.xlabel('y'); plt.ylabel('dy/dt')
plt.title('Plano de fase do experimento do relé: espiral que converge à órbita fechada')
plt.legend(); plt.grid(True)
plt.show()

### 2.2 Prevendo o ciclo no papel: `describing_function_plot`

A biblioteca traz o método gráfico da **função descritiva** pronto (módulo `descfcn`,
demonstrado no exemplo oficial *Describing function analysis* da documentação 0.10.2:
<https://python-control.readthedocs.io/en/0.10.2/examples/describing_functions.html>).
A condição de ciclo-limite $N(a)\,G(j\omega) = -1$ é resolvida **graficamente**: traçam-se a
curva de Nyquist $G(j\omega)$ e a curva $-1/N(a)$; cada **interseção** é um ciclo-limite
previsto — amplitude lida em $-1/N(a)$, frequência lida em $G(j\omega)$.

Para o relé ideal, $-1/N(a) = -\pi a/(4d)$: uma semirreta sobre o eixo real negativo. A
interseção ocorre exatamente onde o Nyquist de $G_3$ cruza esse eixo (fase $-180°$) —
a confirmação geométrica de por que o relé oscila em $\omega_u$:

In [ ]:
# relé ideal como não-linearidade estática (a DF é obtida numericamente pela biblioteca;
# para o relé COM histerese há a versão pronta: ct.relay_hysteresis_nonlinearity(b, c))
def relay_static(x):
    return d * np.copysign(1.0, x)

amp_range = np.linspace(0.005, 0.08, 60)     # faixa de amplitudes candidatas
omega_range = np.logspace(-1, 1.5, 500)

ct.describing_function_plot(G3, relay_static, amp_range, omega_range)
plt.title('Interseção $G_3(j\\omega)$ × $-1/N(a)$: o ciclo-limite previsto no papel')
plt.show()

# comparação: previsão analítica × gráfica × simulação da Seção 2
a_teo = 4 * d / (np.pi * 90.0)
print(f"analítico : a = {a_teo:.4f}, Tu = {2*np.pi/np.sqrt(14):.3f} s")
print(f"simulado  : a = {a:.4f}, Tu = {T_u:.3f} s   (medidos na Seção 2)")
print("gráfico   : leia (a, omega) na interseção da figura acima")

Três caminhos — analítico ($4d/\pi K_u$), gráfico (interseção) e simulação — chegam ao
mesmo ciclo: quando isso acontece, você **entendeu** o fenômeno. No projeto final, o
caminho é o inverso: mede-se $(a, T_u)$ na bancada e recupera-se $K_u$.

---
> **🖼️ Figuras de apoio nos livros:**
> - Ogata, **Figura 8.4** — malha fechada apenas com ganho proporcional para obter $K_{cr}$ (2º método de ZN). Cap. 8, §8.2, **p. 524** (p. 535 do PDF).
> - Ogata, **Figura 8.5** — oscilação sustentada com período $P_{cr}$. Cap. 8, §8.2, **p. 524** (p. 535 do PDF).
> - Penedo, **Figura 9.4** — estratégia de controle do método de Ziegler–Nichols do período crítico (e o método da resposta ao degrau nas figuras do §9.2, pp. 91–92 do PDF). Cap. 9, p. 93 do arquivo PDF.
> - Transparências CDS 110 **L9-1** (29/05/2024), **slide 15** — 'Windup and Anti-Windup Compensation': diagrama de blocos do back-calculation com as respostas com/sem anti-windup.

## Exercícios (relatório do Lab 07)

**E1.** Varie $T_t \in \{0{,}1;\ 0{,}4;\ 1{,}5;\ 5\}$ no anti-windup e compare as respostas.
Existe $T_t$ "bom demais" (que degrade a resposta)? Compare com a regra $T_t = \sqrt{T_i T_d}$
(aqui, sem derivada, use $T_t \approx T_i/2$ como referência).

**E2.** Implemente o anti-windup por **integração condicional** (congelar $\dot x_i$ quando
$u \ne v$ e $e$ tiver o mesmo sinal de $v$) e compare com o back-calculation.

**E3.** Repita o experimento do relé com $d = 0{,}5$ e $d = 2$. As estimativas de $K_u$ e $T_u$
mudam? O que isso diz sobre a robustez do método?

**E4.** Execute o experimento do relé na planta FOPDT identificada no Lab 02
($G(s) = 3e^{-1{,}5s}/(4s+1)$, use Padé). Reporte $K_u$ e $T_u$ para uso no Lab 09.

In [ ]:
# E1 — sua solução aqui

In [ ]:
# E2 — sua solução aqui

In [ ]:
# E3 — sua solução aqui


# **E5.** Reproduza o exemplo oficial da documentação (*Describing function analysis*):
# planta $H(s) = 8/(s^3+2s^2+2s+1)$ em malha fechada com **saturação** unitária
# (`ct.saturation_nonlinearity(1)`). Preveja o ciclo-limite com `describing_function_plot`
# (esperado: $a \approx 3{,}3$, $\omega \approx 1{,}4$ rad/s) e confirme simulando a malha com
# `ct.input_output_response` a partir de uma condição inicial pequena. Por que a saturação
# gera ciclo-limite nessa planta, se a malha linear com ganho 8 é instável? (Dica: pense no
# ganho equivalente $N(a)$ caindo com a amplitude.)

In [ ]:
# E4 — sua solução aqui

In [ ]:
# E5 — sua solução aqui